# MPCount — Colab Setup

**Run cells 1-4 once at the start of every session.** After that, jump straight to Training / Test / Inference.

> Warning: Make sure the runtime is set to GPU: Runtime > Change runtime type > T4 GPU

## Cell 1 - Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Cell 2 - Clone or update the code from GitHub

**First time only**: set FIRST_TIME = True to clone the repo.

**Every session after that**: set FIRST_TIME = False to just pull the latest changes.

Fill in your GitHub username and repo name below.

In [ ]:
import os

GITHUB_USERNAME = 'YOUR_GITHUB_USERNAME'   # <-- change this
REPO_NAME       = 'YOUR_REPO_NAME'          # <-- change this (the repo that contains MPCount)
REPO_URL        = f'https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git'

# Where to keep the code on Drive (persists across sessions)
DRIVE_CODE_DIR  = f'/content/drive/MyDrive/{REPO_NAME}'

FIRST_TIME = True   # <-- set to False after the first clone

if FIRST_TIME:
    !git clone {REPO_URL} {DRIVE_CODE_DIR}
else:
    !git -C {DRIVE_CODE_DIR} pull

# Make this the working directory for the rest of the session
os.chdir(DRIVE_CODE_DIR)
print(f'Working directory: {os.getcwd()}')

## Cell 3 - Install dependencies

Torch and torchvision are already installed by Colab with CUDA support.
We use requirements_colab.txt which skips them to avoid overwriting with a CPU-only version.

In [ ]:
!pip install -r requirements_colab.txt -q
print('Dependencies installed.')

## Cell 4 - Link data from Drive

Your dataset folders (sta, stb, etc.) should live on Drive so they persist.
This cell creates a data/ symlink inside the code folder pointing to them.

Expected Drive layout:
```
MyDrive/
  MPCount_data/
    sta/   <- ShanghaiTech Part A
    stb/   <- ShanghaiTech Part B
```
Adjust DRIVE_DATA_DIR below if your folder is named differently.

In [ ]:
import os

DRIVE_DATA_DIR = '/content/drive/MyDrive/MPCount_data'  # <-- adjust if needed

# Create a symlink so the configs 'data/sta' paths resolve correctly
if not os.path.exists('data'):
    os.symlink(DRIVE_DATA_DIR, 'data')
    print(f'Symlink created: data -> {DRIVE_DATA_DIR}')
else:
    print('data/ already exists (symlink or real folder)')

# Sanity check
print('\nContents of data/:', os.listdir('data'))

## Cell 5 - Verify GPU

Quick sanity check before starting any training.

In [ ]:
import torch
print('Torch version :', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU           :', torch.cuda.get_device_name(0))
    print('VRAM          :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

---
## Training

In [ ]:
# Train on ShanghaiTech Part A
!python main.py --config configs/sta_train_fixed.yml --task train

In [ ]:
# Train on ShanghaiTech Part B
!python main.py --config configs/stb_train.yml --task train

## Testing

In [ ]:
# Test: trained on STA, test on STB
!python main.py --config configs/sta_test_stb.yml --task test

In [ ]:
# Test: trained on STB, test on STA
!python main.py --config configs/stb_test_sta.yml --task test

## Inference on a single image or folder

In [ ]:
IMG_PATH   = '/content/drive/MyDrive/MPCount_data/my_image.jpg'  # or a folder path
MODEL_PATH = 'logs/sta_fixed/best_0.pth'                          # adjust to your checkpoint
VIS_DIR    = 'vis_output'

!python inference.py \
    --img_path   {IMG_PATH} \
    --model_path {MODEL_PATH} \
    --vis_dir    {VIS_DIR} \
    --device     cuda